In [47]:
import serial
import struct
import time

# UART Configuration
PORT = 'COM3'
BAUD_RATE = 115200
MAX_CHUNK_SIZE = 1024
ACK_BYTE = 128

# Initialize serial
ser = serial.Serial(PORT, BAUD_RATE, timeout=1)
time.sleep(2)  # Wait for port to stabilize

def uart_send_data(data: bytes):
    total_sent = 0
    while total_sent < len(data):
        chunk = data[total_sent:total_sent + MAX_CHUNK_SIZE]
        sent = ser.write(chunk)
        total_sent += sent

def uart_receive_data(size: int) -> bytes:
    received = b''
    while len(received) < size:
        chunk = ser.read(min(MAX_CHUNK_SIZE, size - len(received)))
        if chunk:
            received += chunk
    return received

def main():
    # Step 1: Prepare function ID and data
    function_id = 5
    data_to_send = bytearray([i % 256 for i in range(500 * 1024)])  # 5 MB of pattern data
    data_size = len(data_to_send)

    print(f"Sending function ID = {function_id}, Data size = {data_size}")

    # Step 2: Send function ID and data size
    uart_send_data(struct.pack('B', function_id))  # 1 byte
    uart_send_data(struct.pack('>I', data_size))   # 4 bytes, big-endian

    # Step 3: Wait for ACK (128) from FPGA
    ack = uart_receive_data(1)
    if ack[0] != ACK_BYTE:
        print("Did not receive valid ACK from FPGA")
        return
    print("Received ACK from FPGA. Sending data...")

    # Step 4: Send full data
    uart_send_data(data_to_send)
    print("Data sent. Waiting for FPGA to respond...")

    # Step 5: Receive echoed size (4 bytes)
    echo_size_bytes = uart_receive_data(4)
    echo_size = struct.unpack('>I', echo_size_bytes)[0]
    print(f"Received echo size = {echo_size}")

    # Step 6: Send ACK (128) to allow FPGA to echo data
    uart_send_data(struct.pack('B', ACK_BYTE))

    # Step 7: Receive echo data
    echoed_data = uart_receive_data(echo_size)
    print("Echoed data received. Verifying...")

    # Step 8: Compare
    if echoed_data == data_to_send[:echo_size]:
        print("✅ Data match confirmed!")
    else:
        print("❌ Data mismatch detected!")

    ser.close()

if __name__ == "__main__":
    main()



Sending function ID = 5, Data size = 512000
Received ACK from FPGA. Sending data...
Data sent. Waiting for FPGA to respond...
Received echo size = 512000
Echoed data received. Verifying...
✅ Data match confirmed!
